# Goal

Confirm that gradient descent for my model is correct using gradient checking. To carry out gradient checking you need the following:

1. f_x(x, theta) - function to make prediction using concatenated parameters
2. J(x, y, theta) - function to get cost of using concatenated parameters
3. backprop(x, y) - function to return concatenated derivatives
4. grad_check(x, y) - function to get dtheta_approx
5. analyze_grad_descent(x,y) - function to analyze results of gradient check

In [169]:
# Imports
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(seed=69)

In [170]:
# Import data and splitting data into train / cross-validation / test sets
X = np.load("data/players_batch_6_2026-05-25 21:44:14.801126.npy")
Y = np.load("data/results_batch_6_2026-05-25 21:44:14.801174.npy")
X_train, X_mid, Y_train, Y_mid = train_test_split(X, Y, test_size=0.2, random_state=69)
X_cv, X_test, Y_cv, Y_test = train_test_split(X_mid, Y_mid, test_size=0.5, random_state=69)

X_grad_check = X_train[:50]
Y_grad_check = Y_train[:50]

# Scale features
mean = np.mean(X_grad_check, axis=(0,1), keepdims=True)
std = np.std(X_grad_check, axis=(0,1), keepdims=True)
std = np.where(std == 0, 1.0, std)

X_grad_check_scaled = (X_grad_check - mean) / std
print(X_grad_check_scaled.shape)
print(Y_grad_check.shape)

mean = np.mean(X_train, axis=(0,1), keepdims=True)
std = np.std(X_train, axis=(0,1), keepdims=True)
std = np.where(std == 0, 1.0, std)

X_train_scaled = (X_train - mean) / std
X_cv_scaled = (X_cv - mean) / std
X_test_scaled = (X_test - mean) / std

(50, 22, 9)
(50, 121, 1)


In [171]:
# Constants to make working with theta easier
w_1_end_index = 9
b_1_end_index = 10
w_2_end_index = 2672

In [172]:
def condensed_to_params(
    theta: npt.NDArray
):
    """ 
    Function to convert condensed params or derivatives back into their component pararams or derivatives
    
    Args:
        theta (ndarray): a (2793,) array with condensed params or derivatives
        
    Returns:
        w_1 (ndarray): a (1,9) array with first weights / derivative of first weights
        b_1 (scalar): bias / derivative of bias of first layer
        w_2 (ndarray): a (121, 22) array with weights / derivative of weights of second layer
        b_2 (ndarray): a (121, 1) array with bias / derivative of bias of second layer
    """
    w_1 = np.array(theta[0:w_1_end_index]).reshape((1, 9))
    b_1 = theta[b_1_end_index - 1]
    w_2 = np.array(theta[b_1_end_index:w_2_end_index]).reshape((121, 22))
    b_2 = np.array(theta[w_2_end_index:]).reshape((121,1))
    
    return (w_1, b_1, w_2, b_2)

# Condensing Wp, bp, Wt and bt into theta
def params_to_condensed(
    w_p: npt.NDArray,
    b_p: float,
    w_t: npt.NDArray,
    b_t: npt.NDArray
) -> np.ndarray[tuple[2793], np.dtype[np.float64]]:
    """
    Function to condense model parameters into a single vector
    
    Args:
        w_p (ndarray) - (1, 9) array with weights of player portion of network
        b_p (scalar) - bias for player network
        w_t (ndarray) - (121, 22) array with weights of team portion of network
        b_t (ndarray) - (121, 1) array with bias of team portion of network
        
    Returns:
        theta (ndarray) - (2793,) condensed vector
    """
    theta = []
    # Concatenating w_p
    theta.extend(w_p.flatten(order='C'))
    # Concatenating b_p
    theta.append(np.float64(b_p))
    # Concatenating w_t
    theta.extend(w_t.flatten(order='C'))
    # Concatenating b_t
    theta.extend(b_t.flatten(order='C'))
    return np.array(theta)

In [173]:
def f_x(
    x_i: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Function to get prediction of model from input and concatenated parameters
    
    Args:
        x_i (ndarray): a (22, 9) array with training features
        theta (ndarray): a (2793,) array with concatenated parameters
        
    Returns:
        y_pred_i (ndarray): a (121, 1) array with predicted game outcomes
        z_1_i (ndarray): cached z_2_i value
        a_1_i (ndarray): cached a_1_i value
    """
    w_1, b_1, w_2, b_2 = condensed_to_params(theta=theta)
    
    z_1_i = np.matmul(x_i, w_1.T) + b_1
    a_1_i = np.maximum(0, z_1_i)
    z_2_i = np.matmul(w_2, a_1_i) + b_2
    shifted_logits = z_2_i - np.max(z_2_i, axis=0, keepdims=True)
    e_zi = np.exp(shifted_logits)
    y_pred_i = e_zi / np.sum(e_zi, axis=0, keepdims=True)
    
    return (y_pred_i, z_1_i, a_1_i)
    
x_i_example = np.ones((22,9))
# theta = np.array(list(range(2793)))
theta = np.zeros((2793,))
print(f_x(x_i=x_i_example, theta=theta)[0])

[[0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00

In [174]:
def predict(
    X: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Vectorized implementation of forward propagation
    
    Args:
        X (ndarray): a (m, 22, 9) array with m training examples
        theta (ndarray): a (2793,) array with concatenated paramaters
        
    Returns:
        y_pred (ndarray): a (m, 121, 1) array with predicted game outcomes
        Z_1 (ndarray): cached z_1 value
        A_1 (ndarray): cached a_1 value
    """
    w_1, b_1, w_2, b_2 = condensed_to_params(theta=theta)
    
    Z_1 = np.matmul(X, w_1.T) + b_1
    A_1 = np.maximum(0, Z_1)
    Z_2 = np.matmul(w_2, A_1) + b_2
    
    shifted_logits = Z_2 - np.max(Z_2, axis=1, keepdims=True)
    e_z = np.exp(shifted_logits)
    Y_pred = e_z / np.sum(e_z, axis=1, keepdims=True)
    
    return (Y_pred, Z_1, A_1)

X_example = np.array([
        np.ones((22,9))
    ])
Y_pred, _, _ = predict(
    X=X_example,
    theta=np.zeros((2793,))
)
print(Y_pred)

[[[0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826446]
  [0.00826

In [175]:
def J(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Return the cost J given concatenated parameters theta
    
    Args:
        X (ndarray) : a (m, 22, 9) with m training examples
        Y (ndarray) : a (m, 121, 1) with target labels
        theta (ndarray) : a (2793,) array with concatenated parameters
        
    Returns:
        J (scalar): cost
    """
    m = X.shape[0]
    Y_pred, _, _ = predict(X=X, theta=theta)
    J = -np.sum(Y * np.log(Y_pred + 1e-15)) / m
    return J

print(J(
    X=X_grad_check_scaled,
    Y = Y_grad_check,
    theta = rng.random((2793,))
))

6.864057284663554


In [176]:
def backprop(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Get derivatives that will be used by gradient descent to minimize cost function
    
    Args:
        X (ndarray): a (m, 22, 9) array with m training examples
        Y (ndarray): a (m, 121, 1) array with m training output classes
        theta (ndarray): a (2793,) array with concatenated parameters
        
    Returns:
        dtheta (ndarray): a (2793,) array with concatenated derivatives
    """
    m = X.shape[0]
    Y_pred, Z_1, A_1 = predict(
        X=X,
        theta=theta
    )
    _, _, w_2, _ = condensed_to_params(theta=theta)
    
    dZ_2 = Y_pred - Y
    dw_2 = np.mean(np.matmul(dZ_2, np.transpose(A_1, (0, 2, 1))), axis=0)
    db_2 = np.mean(dZ_2, axis=0)
    da_1 = np.matmul(w_2.T, dZ_2)
    dZ_1 = np.where(Z_1 < 0, 0, da_1)
    dw_1 = np.mean(np.matmul(np.transpose(X, (0, 2, 1)), dZ_1).transpose(0, 2, 1), axis=0)
    db_1 = np.mean(np.sum(dZ_1, axis=1)[:, 0])
    
    # dw_1 = np.zeros((1, 9))
    # db_1 = 0
    # dw_2 = np.zeros((121, 22))
    # db_2 = np.zeros((121, 1))
    
    # for i in range(m):
    #     x_i = X[i]
    #     y_i = Y[i]
    #     y_pred_i, z_1_i, a_1_i = f_x(
    #         x_i=x_i,
    #         theta=theta
    #     )
        
    #     dz_2_i = y_pred_i - y_i
    #     dw_2 += np.matmul(dz_2_i, a_1_i.T)
    #     db_2 += dz_2_i
    #     da_1_i = np.matmul(w_2.T, dz_2_i)
    #     dz_1_i = np.where(z_1_i < 0, 0, da_1_i)
    #     dw_1 += np.matmul(x_i.T, dz_1_i).T
    #     db_1 += np.sum(dz_1_i, axis=0)[0]
        
    # dw_1 /= m
    # db_1 /= m
    # dw_2 /= m
    # db_2 /= m
    
    return params_to_condensed(
        w_p=dw_1,
        b_p=db_1,
        w_t=dw_2,
        b_t=db_2
    )
    
    
print(backprop(
    X= X_grad_check,
    Y= Y_grad_check,
    theta = rng.random((2793,))
))

[1.03088787e+002 1.19533178e+000 1.29332562e+000 ... 4.29721976e-001
 5.88859128e-128 6.57973086e-002]


In [177]:
def grad_check(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray
):
    """ 
    Returns dtheta_approx and dtheta together with similarity of the 2 concatenated arrays
    
    Args:
        X (ndarray): a (m, 22, 9) array with m training examples
        Y (ndarray): a (m, 121, 1) array with m target labels
        theta (ndarray): a (2793,) array with concatenated parameters
        
    Return:
        similarity (scalar): similarity of dtheta and dtheta approx
        dtheta (ndarray): a (2793,) array with concatenated derivatives from backprop
        dtheta_approx (ndarray): a (2793,) array with approximated derivatives
    """
    num_params = theta.shape[0]
    dtheta = backprop(
        X=X,
        Y=Y,
        theta=theta
    )
    dtheta_approx = np.zeros((2793,))
    epsilon = 1e-7
    
    for i in range(num_params):
        if i % 500 == 0 or i == num_params - 1:
            print(f"Getting {i}th value in dtheta approx")
            
        theta_right = theta.copy()
        theta_left = theta.copy()
        theta_right[i] += epsilon
        theta_left[i] -= epsilon
        J_theta_right = J(X=X,Y=Y,theta=theta_right)
        J_theta_left = J(X=X,Y=Y,theta=theta_left)
        d_theta_approx_i = (J_theta_right - J_theta_left) / (2 * epsilon)
        dtheta_approx[i] = d_theta_approx_i
        
    print(dtheta) 
    print(dtheta_approx)
    
    distance = np.linalg.norm(dtheta_approx - dtheta)
    similarity_check = distance / (np.linalg.norm(dtheta_approx) + np.linalg.norm(dtheta))
    
    # Interpreting results
    print(f"The difference between dtheta_approx and dtheta is {similarity_check}")
    return (similarity_check, dtheta, dtheta_approx)

In [178]:
def analyze_grad_check(
    X: npt.NDArray,
    Y: npt.NDArray,
    theta: npt.NDArray,
):
    """ 
    Gets approximate dtheta and compares it with dtheta gotten with implementation of gradient descent
    
    Args:
        X (ndarray): a (m, 2, 1) array with m training examples
        Y (ndarray): a (m,) array with labels for each training example
        theta (ndarray): condensed parameters for network
        calculate_gradients: function to return dtheta for parameters
    """
    _, dtheta, dtheta_approx = grad_check(
        X=X,
        Y=Y,
        theta=theta,
    )
    
    num_params = theta.shape[0]
    
    w_1_problem = 0
    b_1_problem = 0
    w_2_problem = 0
    b_2_problem = 0
    
    # Getting total problem for each category
    for i in range(num_params):
        dtheta_value = dtheta[i]
        dtheta_approx_value = dtheta_approx[i]
        difference = (dtheta_approx_value - dtheta_value) ** 2
        
        if 0 <= i < w_1_end_index:
            w_1_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif w_1_end_index <= i < b_1_end_index:
            b_1_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif b_1_end_index <= i < w_2_end_index:
            w_2_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        elif w_2_end_index <= i:
            b_2_problem += difference / (np.abs(dtheta_approx_value) ** 2  + np.abs(dtheta_value) ** 2)
        else:
            print(f"Don't know how to intepret {i}")
            
    print(f"Problem for w_1 is {w_1_problem}")
    print(f"Problem for b_1 is {b_1_problem}")
    print(f"Problem for w_2 is {w_2_problem}")
    print(f"Problem for b_2 is {b_2_problem}")
    
    if w_1_problem > w_2_problem and w_1_problem > b_2_problem:
        start_range = 0
        end_range = 10
        problem_param = "w_1"
    elif w_2_problem > w_1_problem and w_2_problem > b_2_problem:
        start_range = b_1_end_index
        end_range = start_range + 10
        problem_param = "w_2"
    elif b_2_problem > w_1_problem and b_2_problem > w_2_problem:
        start_range = w_2_end_index
        end_range = start_range + 10
        problem_param = "b_2"
    else:
        start_range = 0
        end_range = 11
        problem_param = "b_1"
    
    for idx in range(start_range, end_range):
        dtheta_value = dtheta[idx]
        dtheta_approx_value = dtheta_approx[idx]
        
        print(f"Element index {idx} in {problem_param} has dtheta of {dtheta_value} and dtheta_approx of {dtheta_approx_value}")

In [179]:
analyze_grad_check(
    X=X_grad_check_scaled,
    Y=Y_grad_check,
    theta=rng.random(2793,)
)

Getting 0th value in dtheta approx
Getting 500th value in dtheta approx
Getting 1000th value in dtheta approx
Getting 1500th value in dtheta approx
Getting 2000th value in dtheta approx
Getting 2500th value in dtheta approx
Getting 2792th value in dtheta approx
[ 1.2103853   0.70604476 -0.31199553 ...  0.00309539  0.00896243
  0.00576519]
[ 1.2103853   0.70604476 -0.31199554 ...  0.00309539  0.00896244
  0.0057652 ]
The difference between dtheta_approx and dtheta is 2.72722797457479e-08
Problem for w_1 is 2.8309625263749396e-16
Problem for b_1 is 4.893134689570681e-18
Problem for w_2 is 3.160637856323685e-08
Problem for b_2 is 1.0035683913437785e-10
Element index 10 in w_2 has dtheta of -0.03353034749978914 and dtheta_approx of -0.03353034738751148
Element index 11 in w_2 has dtheta of -0.08063926132047129 and dtheta_approx of -0.08063926415502465
Element index 12 in w_2 has dtheta of 0.006355753240929705 and dtheta_approx of 0.006355751480668914
Element index 13 in w_2 has dtheta of -